# Estratégia Teacher - Student
A estratégia baseia-se nos princípios de Knowledge Distillation (Destilação de Conhecimento) e Weak Supervision (Supervisão Fraca). Dado que o dataset do CFPB não possui ground truth (rótulo real) de sentimento, utilizamos um Large Language Model (LLM) pré-treinado em Sentiment Analysis (o "Teacher", ex: RoBERTa-Large ou DeBERTa) para realizar uma inferência Zero-Shot ou Few-Shot sobre o corpus não rotulado.

O modelo Teacher, devido à sua alta complexidade paramétrica (milhões de parâmetros), possui capacidade semântica superior para desambiguar contextos financeiros, gerando Pseudo-Labels (rótulos artificiais) com alta confiança. Estes rótulos, refinados por regras determinísticas baseadas em metadados (como a resposta da empresa ), constituem o Silver Standard Target. O modelo final ("Student", ex: DistilBERT ou MLP), que possui uma arquitetura otimizada para inferência (baixa latência/custo computacional), é então treinado supervisionadamente para minimizar a função de perda (loss function) em relação a essa distribuição de probabilidades gerada pelo Teacher, efetivamente "aprendendo" a generalização semântica complexa dentro de uma estrutura compacta.

Imagine que você tem uma pilha gigante de 1 milhão de cartas de clientes e precisa classificar se elas são "Elogios" ou "Reclamações", mas você não tem tempo para ler todas.

A estratégia Teacher-Student funciona assim:

1) O Professor (Teacher): Você contrata um especialista muito experiente (um modelo de IA gigante e "inteligente", mas lento e caro de rodar). Ele lê uma parte dessas cartas e coloca um carimbo: "Positivo" ou "Negativo". Como ele já leu bilhões de textos na internet, ele entende sarcasmo, gírias e contextos complexos.

2) O Aluno (Student): Você pega um estagiário rápido e dedicado (um modelo de IA menor e mais leve). O estagiário não sabe nada no começo. Ele pega as cartas que o Professor já carimbou e estuda: "Ah, quando tem essa palavra, o Mestre marcou como Negativo".

3) O Objetivo: O estagiário treina tanto que aprende a imitar as decisões do Mestre. No final, você demite o Mestre (que é caro) e fica só com o estagiário, que agora consegue classificar novas cartas muito rápido e com a mesma qualidade do especialista.

No nosso projeto, o "Professor" cria a coluna de Sentimento (que não existe no arquivo original ) e o "Aluno" será o modelo final que entregaremos.

[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [1]:
import gc
import os
from pathlib import Path
from typing import Generator, Tuple, List
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.nn.functional import softmax
from tqdm.auto import tqdm
import polars as pl
import torch
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')


In [2]:
# ==========================================
# CONFIGURAÇÃO
# ==========================================
BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / "data" / "parquet" / "complaints_lean_2025.parquet"
OUTPUT_DIR = BASE_DIR / "data" / "parquet" / "labeled"

# Ajuste conforme VRAM
GPU_BATCH_SIZE = 12 
# Tamanho do bloco de leitura (RAM)
IO_CHUNK_SIZE = 5000 
# A cada quantas linhas salvar um arquivo parcial
SAVE_EVERY_N_ROWS = 50000

In [3]:
# ==========================================
# 1. FUNÇÕES DE CHECKPOINT (NOVO)
# ==========================================
def get_checkpoint_state(output_dir: Path) -> Tuple[int, int]:
    """
    Verifica a pasta de saída para descobrir onde paramos.
    Retorna: (total_linhas_processadas, proximo_chunk_id)
    """
    if not output_dir.exists():
        return 0, 0
    
    # Lista arquivos existentes padrão: labeled_part_X.parquet
    files = list(output_dir.glob("labeled_part_*.parquet"))
    
    if not files:
        return 0, 0
    
    print("Verificando checkpoint (contando linhas processadas)...")
    
    # 1. Descobrir o próximo ID de chunk
    max_id = -1
    for f in files:
        # Extrai o número do arquivo (ex: labeled_part_12.parquet -> 12)
        match = re.search(r"labeled_part_(\d+).parquet", f.name)
        if match:
            max_id = max(max_id, int(match.group(1)))
    
    next_chunk_id = max_id + 1
    
    # 2. Contar linhas totais já salvas (Rápido com Scan)
    # Usamos o scan_parquet com glob para ler metadados de todos os arquivos
    try:
        total_processed = pl.scan_parquet(output_dir / "labeled_part_*.parquet").select(pl.len()).collect().item()
    except Exception as e:
        print(f"Aviso: Erro ao ler checkpoint ({e}). Recomeçando do zero por segurança.")
        return 0, 0
        
    return total_processed, next_chunk_id

In [4]:
# ==========================================
# 2. DATA LOADER COM OFFSET (ATUALIZADO)
# ==========================================
class DataLoaderEfficient:
    def __init__(self, input_path: Path, io_chunk_size: int):
        self.input_path = input_path
        self.io_chunk_size = io_chunk_size
        
        if not self.input_path.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {self.input_path}")
            
        self.lazy_df = pl.scan_parquet(self.input_path)
        self.total_rows = self.lazy_df.select(pl.len()).collect().item()

    def get_total_rows(self) -> int:
        return self.total_rows

    def stream_data(self, start_offset: int = 0) -> Generator[Tuple[List[str], List[str]], None, None]:
        """
        Lê dados em chunks a partir de um deslocamento inicial (Resume).
        """
        current_offset = start_offset
        
        print(f"DataLoader: Iniciando leitura a partir da linha {current_offset:,}...")
        
        while current_offset < self.total_rows:
            # Fatia o arquivo original (Slice é Lazy e eficiente)
            # offset, length
            df_chunk = self.lazy_df.slice(current_offset, self.io_chunk_size).collect()
            
            # Avança o ponteiro
            current_offset += self.io_chunk_size
            
            # Filtra linhas válidas (com texto e flag True)
            df_clean = df_chunk.filter(
                (pl.col("consumer_complaint_narrative").is_not_null()) &
                (pl.col("has_narrative") == True)
            ).select(["complaint_id", "consumer_complaint_narrative"])

            if df_clean.height > 0:
                yield df_clean["complaint_id"].to_list(), df_clean["consumer_complaint_narrative"].to_list()

In [5]:
# ==========================================
# 3. TEACHER (IDÊNTICO AO ANTERIOR)
# ==========================================
class SentimentTeacher:
    def __init__(self, model_name: str, batch_size: int):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        print(f"--- Carregando Teacher no dispositivo: {self.device} ---")
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()
        
        if self.device.type == 'cuda':
            self.model.half()

    def predict_batch(self, texts: List[str]) -> List[dict]:
        results = []
        for i in range(0, len(texts), self.batch_size):
            batch_texts = texts[i : i + self.batch_size]
            inputs = self.tokenizer(
                batch_texts, return_tensors="pt", padding=True, 
                truncation=True, max_length=512
            ).to(self.device)
            
            with torch.no_grad():
                outputs = self.model(**inputs)
                probs = softmax(outputs.logits, dim=-1)
            
            probs_cpu = probs.float().cpu().numpy()
            predicted_indices = np.argmax(probs_cpu, axis=1)
            
            # Mapping do Modelo RoBERTa
            # 0 -> Negative, 1 -> Neutral, 2 -> Positive
            labels_map = {0: "Negative", 1: "Neutral", 2: "Positive"}
            
            for idx, pred_idx in enumerate(predicted_indices):
                results.append({
                    "teacher_label": labels_map[pred_idx],
                    "teacher_score": float(probs_cpu[idx][pred_idx]),
                    "score_neg": float(probs_cpu[idx][0]),
                    "score_neu": float(probs_cpu[idx][1]),
                    "score_pos": float(probs_cpu[idx][2])
                })
            
            del inputs, outputs, probs
            if self.device.type == 'cuda':
                torch.cuda.empty_cache()
        return results

In [6]:
# ==========================================
# 4. ORQUESTRADOR COM RESUME
# ==========================================
def save_chunk(data: List[dict], output_dir: Path, chunk_id: int):
    filename = output_dir / f"labeled_part_{chunk_id}.parquet"
    pl.DataFrame(data).write_parquet(filename)
    # print(f"Chunk {chunk_id} salvo.")

def run_pipeline():
    if not INPUT_FILE.exists():
        print(f"ERRO: Arquivo de entrada não encontrado em {INPUT_FILE}")
        return

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # --- LÓGICA DE RETOMADA (CHECKPOINT) ---
    rows_processed, next_chunk_id = get_checkpoint_state(OUTPUT_DIR)
    
    if rows_processed > 0:
        print(f"RESUME: Encontrados {rows_processed:,} registros já processados.")
        print(f"Continuando a partir do Chunk ID: {next_chunk_id}")
    else:
        print("INÍCIO: Nenhum checkpoint encontrado. Começando do zero.")

    # Inicialização
    loader = DataLoaderEfficient(INPUT_FILE, io_chunk_size=IO_CHUNK_SIZE)
    teacher = SentimentTeacher("cardiffnlp/twitter-roberta-base-sentiment-latest", GPU_BATCH_SIZE)
    
    total_rows = loader.get_total_rows()
    remaining_rows = total_rows - rows_processed
    
    if remaining_rows <= 0:
        print("Todos os registros já foram processados! Nada a fazer.")
        return

    print(f"Processando {remaining_rows:,} registros restantes...")
    
    # Loop Principal
    results_buffer = []
    chunk_counter = next_chunk_id # Começa do número correto
    
    pbar = tqdm(total=remaining_rows, desc="Teacher Inference", unit="rows")
    
    try:
        # Passamos rows_processed como offset inicial
        for ids, texts in loader.stream_data(start_offset=rows_processed):
            
            # Inferência
            predictions = teacher.predict_batch(texts)
            
            # Merge
            for cid, pred in zip(ids, predictions):
                pred["complaint_id"] = cid
                results_buffer.append(pred)
            
            pbar.update(len(texts))
            
            # Salvar Parcialmente
            if len(results_buffer) >= SAVE_EVERY_N_ROWS:
                save_chunk(results_buffer, OUTPUT_DIR, chunk_counter)
                chunk_counter += 1
                results_buffer = []
                gc.collect()
                
    except KeyboardInterrupt:
        print("\nInterrompido pelo usuário! Salvando buffer atual antes de sair...")
    
    # Salvar o que sobrou no buffer
    if results_buffer:
        save_chunk(results_buffer, OUTPUT_DIR, chunk_counter)
        
    pbar.close()
    print(f"\nPipeline concluído!")

In [7]:
# ==========================================
# EXECUÇÃO
# ==========================================
if __name__ == "__main__":
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    run_pipeline()

Verificando checkpoint (contando linhas processadas)...
RESUME: Encontrados 1,192,432 registros já processados.
Continuando a partir do Chunk ID: 24
--- Carregando Teacher no dispositivo: cuda ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Todos os registros já foram processados! Nada a fazer.


[Voltar para notebook principal](./0_notebook_principal.ipynb)